# FinQA 数据资产准备

本 Notebook 为银行金融大模型量化项目准备三类资产：

- `calibration`：用于 AWQ/GPTQ/SmoothQuant 校准；
- `quality_dev` 和 `quality_test`：用于质量评估；
- `high_risk_regression`：固定的金融数值回归集。

数据会缓存到 Google Drive，避免 Colab 运行时重置后重复下载。

In [ ]:
!pip -q install -U "datasets>=2.19,<4.0" huggingface_hub pandas pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 载入项目源码

如果你已经把项目目录上传到 Colab，请修改 `PROJECT_DIR`。如果项目放在 Git 仓库，可以填写 `REPO_URL` 自动克隆。

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = '/content/OptimizationSystem'
REPO_URL = ''

if REPO_URL and not Path(PROJECT_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)

if not Path(PROJECT_DIR).exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} 不存在，请上传项目或填写 REPO_URL。'
    )

sys.path.insert(0, PROJECT_DIR)

In [ ]:
from src.data.finqa_assets import prepare_assets

OUTPUT_DIR = (
    '/content/drive/MyDrive/'
    'banking_llm_project/datasets'
)

assets = prepare_assets(
    output_dir=OUTPUT_DIR,
    calibration_size=1024,
    seed=42,
)

assets.summary()

In [ ]:
print('校准集:', len(assets.calibration))
print('质量调优集:', len(assets.quality_dev))
print('最终质量评估集:', len(assets.quality_test))
print('高风险回归集:', len(assets.high_risk_regression))
print('第一条校准 Prompt:')
print(assets.calibration_prompts[0][:2000])

## 接入量化算法

`assets.calibration_prompts` 可以直接传给你的 AWQ/GPTQ 校准流程。不要把答案字段拼进校准 Prompt。

In [ ]:
calibration_prompts = assets.calibration_prompts

for prompt in calibration_prompts[:3]:
    print(prompt[:500])
    print('---')

In [ ]:
from src.data.finqa_assets import load_assets

reloaded_assets = load_assets(OUTPUT_DIR)
reloaded_assets.summary()

## 使用规则

1. 用 `calibration` 计算量化参数。
2. 用 `quality_dev` 选择 group size、敏感层和混合精度。
3. 配置冻结后才使用 `quality_test` 生成最终报告。
4. 每个 FP16、INT8、INT4、剪枝或后端版本都运行 `high_risk_regression`。
5. 重点统计“FP16 正确但压缩模型错误”的新增退化。